In [ ]:
!pip -q install earthengine-api
!gcloud projects list

In [ ]:
import ee
import pandas as pd
import matplotlib.pyplot as plt

ee.Authenticate()                 # <-- uncomment on the FIRST run only
ee.Initialize(project="ee-sanjusajimon76")   # <-- your Earth Engine project id

WIN = 3   # half-window (months) for compositing windows & change detection

In [ ]:
# %% ---- 1. Area of interest (from the provided shapefile) --------------------
# Upload the shapefile as a GEE asset, then put its id here:
AOI_ASSET = "projects/ee-sanjusajimon76/assets/study_area"   # <-- EDIT THIS

aoi = ee.FeatureCollection(AOI_ASSET)
region = aoi.geometry()

# Sanity check — if this prints a microscopic area, your AOI is wrong.
print("AOI area (km^2):", round(region.area(maxError=1).divide(1e6).getInfo(), 4))

# --- Fallback ONLY if you have no asset yet (replace with your real coords;
#     order is [west, south, east, north], i.e. xmin,ymin,xmax,ymax):
# region = ee.Geometry.Rectangle([-73.66, 1.88, -73.62, 1.92])



In [ ]:

# %% ---- 2. Cloud masking -----------------------------------------------------
def mask_s2_sr(img):
    """Sentinel-2 SR: drop cloud/shadow/cirrus/snow using the Scene Class Layer."""
    scl = img.select("SCL")
    clear = (scl.neq(3)      # 3  = cloud shadow
             .And(scl.neq(8))   # 8  = cloud (medium prob)
             .And(scl.neq(9))   # 9  = cloud (high prob)
             .And(scl.neq(10))  # 10 = cirrus
             .And(scl.neq(11))) # 11 = snow/ice
    return img.updateMask(clear)


def prep_l8_sr(img):
    """Landsat-8 C2 L2: mask cloud/shadow via QA_PIXEL, then apply SR scale factors."""
    qa = img.select("QA_PIXEL")
    cloud  = qa.bitwiseAnd(1 << 3).neq(0)   # bit 3 = cloud
    shadow = qa.bitwiseAnd(1 << 4).neq(0)   # bit 4 = cloud shadow
    masked = img.updateMask(cloud.Or(shadow).Not())
    optical = masked.select("SR_B.").multiply(0.0000275).add(-0.2)
    return masked.addBands(optical, None, True)

In [ ]:

# %% ---- 3. NDVI per sensor ---------------------------------------------------
def ndvi_s2(img):
    return img.addBands(img.normalizedDifference(["B8", "B4"]).rename("NDVI"))   # NIR/Red

def ndvi_l8(img):
    return img.addBands(img.normalizedDifference(["SR_B5", "SR_B4"]).rename("NDVI"))  # NIR/Red

s2 = (ee.ImageCollection("COPERNICUS/S2_SR_HARMONIZED")
        .filterBounds(region)
        .filterDate("2019-01-01", "2024-12-31")
        .filter(ee.Filter.lt("CLOUDY_PIXEL_PERCENTAGE", 60))
        .map(mask_s2_sr).map(ndvi_s2).select("NDVI"))

l8 = (ee.ImageCollection("LANDSAT/LC08/C02/T1_L2")
        .filterBounds(region)
        .filterDate("2014-01-01", "2018-12-31")
        .map(prep_l8_sr).map(ndvi_l8).select("NDVI"))



In [ ]:
# %% ---- 4. Monthly median composites (robust to empty months) ----
def monthly(coll, start, n_months):
    """One median NDVI image per month. Empty months -> a fully-masked NDVI band,
    so the 'NDVI' key always exists and reduceRegion won't error."""
    start = ee.Date(start)
    empty = ee.Image.constant(0).rename("NDVI").updateMask(ee.Image.constant(0))
    def one(m):
        s = start.advance(m, "month")
        e = s.advance(1, "month")
        sub = coll.filterDate(s, e)
        img = ee.Image(ee.Algorithms.If(sub.size().gt(0), sub.median(), empty))
        return img.rename("NDVI").set("system:time_start", s.millis())
    return ee.ImageCollection(ee.List.sequence(0, n_months - 1).map(one))

l8_m = monthly(l8, "2014-01-01", 60)
s2_m = monthly(s2, "2019-01-01", 72)

In [ ]:
 #%% ---- 5. AOI-mean NDVI time series (per sensor, kept separate) -------------
def to_series(coll, sensor, scale):
    def feat(img):
        v = img.reduceRegion(ee.Reducer.mean(), region,
                             scale=scale, maxPixels=int(1e13)).get("NDVI")
        return ee.Feature(None, {"date": img.date().format("YYYY-MM-dd"),
                                 "ndvi": v, "sensor": sensor})
    fc = coll.map(feat).filter(ee.Filter.notNull(["ndvi"]))
    rows = fc.getInfo()["features"]
    return pd.DataFrame([r["properties"] for r in rows])

df_l8 = to_series(l8_m, "Landsat-8", 30)
df_s2 = to_series(s2_m, "Sentinel-2", 30)
df = pd.concat([df_l8, df_s2], ignore_index=True)
df["date"] = pd.to_datetime(df["date"])
df = df.sort_values("date").reset_index(drop=True)
print("valid months  | Landsat-8:", len(df_l8), " Sentinel-2:", len(df_s2))


In [ ]:


# %% ---- 6. Sustained-change detection (within each sensor, true calendar months)
def find_break(d):
    """Strongest *sustained* NDVI decline: mean(next WIN months) - mean(prev WIN months)."""
    s = d.set_index("date").sort_index()["ndvi"].resample("MS").mean()  # monthly calendar
    smooth = s.rolling(WIN, center=True, min_periods=1).mean()
    before = smooth.rolling(WIN, min_periods=2).mean()
    after  = smooth.shift(-WIN).rolling(WIN, min_periods=2).mean()
    sustained = after - before
    if sustained.notna().any():
        t = sustained.idxmin()
        return pd.Timestamp(t), float(sustained.min())
    return None, None

breaks = {}
for sensor, g in df.groupby("sensor"):
    bdate, drop = find_break(g)
    breaks[sensor] = (bdate, drop)
    if bdate is not None:
        print(f"{sensor}: strongest sustained drop ~ {bdate.date()}  (delta NDVI = {drop:.3f})")
    else:
        print(f"{sensor}: no sustained drop found")

# pick the most negative drop across sensors
break_sensor = min(breaks, key=lambda k: (breaks[k][1] if breaks[k][1] is not None else 0.0))
break_date, break_drop = breaks[break_sensor]
print(f"\nSelected break -> {break_sensor}  {break_date.date()}  (delta NDVI = {break_drop:.3f})")


In [ ]:


# %% ---- 7. Plot the time series (sensors visually separate — never one trend)
plt.figure(figsize=(11, 4.5))
for sensor, g in df.groupby("sensor"):
    g = g.sort_values("date")
    plt.plot(g["date"], g["ndvi"], marker="o", ms=3, lw=1, label=sensor)
plt.axvline(break_date, color="crimson", ls="--", lw=1.2,
            label=f"sustained drop ~ {break_date.date()}")
plt.ylim(0, 1); plt.ylabel("Mean NDVI over AOI"); plt.xlabel("Year")
plt.title("NDVI time series  (Landsat-8 and Sentinel-2 shown separately)")
plt.legend(); plt.grid(alpha=0.3); plt.tight_layout(); plt.show()
# NOTE: any offset between the two coloured series is a sensor difference,
# NOT real vegetation change. Read each sensor's trend on its own.


In [ ]:

# %% ---- 8. Before/after composites (SAME sensor as the break) + change map ---
src   = s2_m if break_sensor == "Sentinel-2" else l8_m
bdate = ee.Date(break_date.strftime("%Y-%m-%d"))

before_img = src.filterDate(bdate.advance(-WIN, "month"), bdate).median().clip(region)
after_img  = src.filterDate(bdate, bdate.advance(WIN, "month")).median().clip(region)
ndvi_change = after_img.subtract(before_img).rename("NDVI_change").clip(region)



In [ ]:


# %% ---- 9. Interactive map ---------------------------------------------------
import geemap
Map = geemap.Map()
Map.centerObject(region, 13)
ndvi_vis   = {"min": 0,    "max": 1,   "palette": ["#b5651d", "#e0c060", "#9ad36b", "#2f7d4a"]}
change_vis = {"min": -0.4, "max": 0.4, "palette": ["#d7191c", "#ffffbf", "#1a9641"]}  # red = loss
Map.addLayer(before_img.select("NDVI"), ndvi_vis, f"NDVI before ({break_sensor})")
Map.addLayer(after_img.select("NDVI"),  ndvi_vis, f"NDVI after ({break_sensor})")
Map.addLayer(ndvi_change, change_vis, "NDVI change (after - before)")
Map.addLayer(ee.Image().paint(aoi, 0, 2), {"palette": ["#1f77ff"]}, "AOI outline")
Map   # shows inline in a notebook


In [ ]:


# %% ---- 10. Export GeoTIFFs to Drive ----------------------------------------
scale = 10 if break_sensor == "Sentinel-2" else 30
for name, img in [("NDVI_before", before_img.select("NDVI")),
                  ("NDVI_after",  after_img.select("NDVI")),
                  ("NDVI_change", ndvi_change)]:
    ee.batch.Export.image.toDrive(
        image=img, description=name, folder="GEE_NDVI_Exports",
        region=region, scale=scale, maxPixels=int(1e13), fileFormat="GeoTIFF"
    ).start()
    print("export started:", name)


In [ ]:
 %% ---- 11. Honest reporting -------------------------------------------------
# - Report the strongest *sustained* decline (date + delta NDVI), per sensor.
# - Say the result is "consistent with vegetation loss / a candidate event",
#   and recommend confirmation — do NOT declare deforestation from NDVI alone.
# - State limitations: tropical cloud cover, S2/L8 are not directly comparable,
#   NDVI saturates over dense canopy, and everything is clipped to the real AOI.
